In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import randint, uniform
import joblib

SEED = 42
np.random.seed(SEED)

# Загружаем данные
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

In [7]:
# Определяем колонки (числовые и категориальные)
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Препроцессор с обработкой пропусков
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

/var/folders/t9/w50p8w0917b0t61ks_6y_3g40000gn/T/ipykernel_40804/2432141586.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


In [8]:
# Модель GradientBoostingRegressor
gb = GradientBoostingRegressor(random_state=SEED)

# Пайплайн
pipe = Pipeline([
    ('prep', preprocessor),
    ('reg', gb)
])

# Сетка гиперпараметров для перебора
param_dist = {
    'reg__n_estimators': randint(50, 300),          # количество деревьев
    'reg__max_depth': randint(3, 10),               # глубина деревьев
    'reg__learning_rate': uniform(0.01, 0.3),       # скорость обучения
    'reg__subsample': uniform(0.6, 0.4),            # доля выборки для каждого дерева
    'reg__min_samples_split': randint(2, 20),       # мин. выборки для разбиения
    'reg__min_samples_leaf': randint(1, 10)         # мин. выборки в листе
}

# RandomizedSearchCV с 3-кратной кросс-валидацией (на train)
random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=30,              # количество комбинаций (можно увеличить до 50-100 при большом времени)
    cv=3,
    scoring='r2',
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

In [9]:
print("Начинаем подбор гиперпараметров для GradientBoosting...")
random_search.fit(X_train, y_train)

print("\nЛучшие параметры:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nЛучшее R² (CV): {random_search.best_score_:.4f}")

Начинаем подбор гиперпараметров для GradientBoosting...
Fitting 3 folds for each of 30 candidates, totalling 90 fits


KeyboardInterrupt: 